In [10]:
import fsspec
import xarray as xr
import scipy.spatial
import numpy as np
import os
import argparse
from datetime import date
import datetime
from calculations.calculations import vapor_pressure
from calculations.calculations import wind_tot
from calculations.calculations import rel_hum
import regionmask
import geopandas as gpd

##### Function

In [2]:
# Using code from https://github.com/google-research/arco-era5/blob/main/docs/0-Surface-Reanalysis-Walkthrough.ipynb 
def build_triangulation(lon, lat):
    """
    Creates a Delaunay tesselation
    
    """
    lon_grid, lat_grid = np.meshgrid(lon, lat)
    grid = np.stack([lon_grid.ravel(), lat_grid.ravel()], axis=1)

    return scipy.spatial.Delaunay(grid)

def interpolate(data, tri, mesh):
    """
    Interpolates the ERA5 grid using the Delaunay tesselation
    
    """
    indices = tri.find_simplex(mesh) 
    ndim = tri.transform.shape[-1]
    T_inv = tri.transform[indices, :ndim, :]
    r = tri.transform[indices, ndim, :]
    c = np.einsum('...ij,...j', T_inv, mesh - r)
    c = np.concatenate([c, 1 - c.sum(axis=-1, keepdims=True)], axis=-1)
    # result = np.einsum('...i,...i', data[:, tri.simplices[indices]], c)
    dat = data.reshape(8760,783)
    result = np.einsum('t...i,...i->t...', dat[:, tri.simplices[indices]], c)
    return np.where(indices == -1, np.nan, result)

### Main

In [9]:
fs = fsspec.filesystem('gs')
fs.ls('gs://gcp-public-data-arco-era5/co/')

# Opening dataset with zarr
reanalysis = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3', 
    chunks={'time': 48},
    consolidated=True,
    )

In [11]:
reanalysis

<xarray.Dataset> Size: 4PB
Dimensions:                                                          (
                                                                      time: 1323648,
                                                                      latitude: 721,
                                                                      longitude: 1440,
                                                                      level: 37)
Coordinates:
  * latitude                                                         (latitude) float32 3kB ...
  * level                                                            (level) int64 296B ...
  * longitude                                                        (longitude) float32 6kB ...
  * time                                                             (time) datetime64[ns] 11MB ...
Data variables: (12/273)
    100m_u_component_of_wind                                         (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    100m_v_component_of_wind                                         (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    10m_u_component_of_neutral_wind                                  (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    10m_u_component_of_wind                                          (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    10m_v_component_of_neutral_wind                                  (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    10m_v_component_of_wind                                          (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    ...                                                               ...
    wave_spectral_directional_width_for_swell                        (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    wave_spectral_directional_width_for_wind_waves                   (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    wave_spectral_kurtosis                                           (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    wave_spectral_peakedness                                         (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    wave_spectral_skewness                                           (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
    zero_degree_level                                                (time, latitude, longitude) float32 5TB dask.array<chunksize=(48, 721, 1440), meta=np.ndarray>
Attributes:
    last_updated:           2025-11-11 01:57:00.119987+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-04-30
    valid_time_stop_era5t:  2025-11-05

In [4]:
# # Load GeoJSON file into a GeoDataFrame
# gdf = gpd.read_file('../county_boundaries/illinois_counties.geojson')

# # Display the first few rows to see structure and attributes
# print(gdf.head())
# # Print column names (usually includes 'geometry' plus properties)
# print(gdf.columns)
# # Check the geometry types (Point, Polygon, etc.)
# print(gdf.geom_type.unique()) -->

In [3]:
# (gdf['NAME'].unique())

In [4]:
# Define vars to be called 
variable = ['2m_temperature','10m_u_component_of_wind','10m_v_component_of_wind','2m_dewpoint_temperature','total_precipitation'] #EDIT /data/keeling/a/rytam2/a/iema_output/variables_202510210102.nc
# calc = #EDIT 
year_start = 2016 #EDIT 
year_end = 2025 #EDIT 

# Define Once
# Dates
i_date = str(year_start) + '-01-01'
f_date = str(year_end)   + '-12-31'


recent_an = reanalysis.sel(time=slice(i_date, f_date))
era5_var = recent_an[variable]

lon_min = 267.2
lon_max = 274
lat_min = 36
lat_max = 43.5

illinois_ds = era5_var.where(
(recent_an.longitude > lon_min) & (recent_an.latitude > lat_min) &
(recent_an.longitude < lon_max) & (recent_an.latitude < lat_max),
drop=True).rename({'longitude':'lon', 'latitude':'lat'})


## Creating County Coords
# Load county boundaries
counties = gpd.read_file('/data/keeling/a/rytam2/iema/run-only-climate_map/climate_map/county_boundaries/illinois_counties.geojson')
# Create a mask for each county using the DataFrame index instead of looking for an 'index' column
# The default behavior (numbers=None) will use the position in the DataFrame
masks = regionmask.mask_geopandas(counties, illinois_ds.lon, illinois_ds.lat)
# Convert the mask to a DataArray with county names
county_names = counties["NAME"].values
county_indices = masks.values


# Create a county name array with the same shape as the mask
county_array = np.full(county_indices.shape, "", dtype="object")
# Assign county names based on indices
for i, name in enumerate(county_names):
    county_array[county_indices == i] = name
# Set cells outside any county to NaN
county_array[county_indices == -1] = np.nan
# Add as a coordinate to the dataset
county_da = xr.DataArray(
    county_array, dims=["lat", "lon"], coords={"lat": illinois_ds.lat, "lon": illinois_ds.lon}
)

# # Pulling appropriate variables for calculations        
# if variable == 'vapor_pressure':
#     variable = '2m_dewpoint_temperature'
#     calc = 'vapor_pressure'
# elif variable == 'sfcWind':
#     variable = ['10m_u_component_of_wind','10m_v_component_of_wind']
#     calc = 'sfcWind'
# elif variable == 'relative_humidity':
#     variable = ['2m_temperature','2m_dewpoint_temperature']
#     calc = 'relative_humidity'

# # Calculations
# if calc=='vapor_pressure':
#     fin_array = vapor_pressure(fin_array) # Calculation
# elif calc=='sfcWind':
#     fin_array,_ = wind_tot(fin_array['10m_u_component_of_wind'], fin_array['10m_v_component_of_wind'])
# elif calc=='relative_humidity':
#     fin_array = rel_hum(fin_array['2m_dewpoint_temperature'], fin_array['2m_temperature'])

# only apply interp to unstructrued ERA5 (); otherwise keep regular grid 
# ds_dict = {}
dataset = 'regridded' # 'regridded' or 'raw'
# for i in variable:     
if dataset == 'raw':
    tri = build_triangulation(illinois_ds.longitude, illinois_ds.latitude)
    longitude = np.linspace(lon_min, lon_max, num=round(lon_max-lon_min)*4+1)
    latitude = np.linspace(lat_min, lat_max, num=round(lat_max-lat_min)*4+1)
        
    mesh = np.stack(np.meshgrid(longitude, latitude, indexing='ij'), axis=-1)
    mesh_int = interpolate(illinois_ds[variable].values, tri, mesh)
    
    fin_array = xr.DataArray(da.from_array(mesh_int, chunks=(48, 29, 33)), 
                 coords=[('time', illinois_ds.time.data), ('lon', longitude), ('lat', latitude)]).rename(variable).assign_coords(county=county_da)
else:
    fin_array = illinois_ds.assign_coords(county=county_da)
    # ds_dict[i] = fin_array

In [7]:
t2m = fin_array[variable[0]]
u = fin_array[variable[1]]
v = fin_array[variable[2]]
dp = fin_array[variable[3]]
precip = fin_array[variable[4]]

In [8]:
precip

<xarray.DataArray 'total_precipitation' (time: 87672, lat: 29, lon: 27)> Size: 275MB
dask.array<where, shape=(87672, 29, 27), dtype=float32, chunksize=(48, 29, 27), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
  * time     (time) datetime64[ns] 701kB 2016-01-01 ... 2025-12-31T23:00:00
    county   (lat, lon) object 6kB '' '' '' '' '' '' '' ... '' '' '' '' '' '' ''
Attributes:
    long_name:   Total precipitation
    short_name:  tp
    units:       m

### Save

In [26]:
ds = xr.Dataset({'t2m':t2m, 'u':u, 'v':v, 'dp':dp, 'precip':precip})
ds

<xarray.Dataset> Size: 1GB
Dimensions:  (lat: 29, lon: 27, time: 87672)
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
  * time     (time) datetime64[ns] 701kB 2016-01-01 ... 2025-12-31T23:00:00
    county   (lat, lon) object 6kB '' '' '' '' '' '' '' ... '' '' '' '' '' '' ''
Data variables:
    t2m      (time, lat, lon) float32 275MB dask.array<chunksize=(48, 29, 27), meta=np.ndarray>
    u        (time, lat, lon) float32 275MB dask.array<chunksize=(48, 29, 27), meta=np.ndarray>
    v        (time, lat, lon) float32 275MB dask.array<chunksize=(48, 29, 27), meta=np.ndarray>
    dp       (time, lat, lon) float32 275MB dask.array<chunksize=(48, 29, 27), meta=np.ndarray>
    precip   (time, lat, lon) float32 275MB dask.array<chunksize=(48, 29, 27), meta=np.ndarray>

In [29]:
filename = '/data/keeling/a/rytam2/a/iema_output/variables_'+datetime.now().strftime("%Y%m%d%H%M")+'.nc'
ds.to_netcdf(filename)
print(filename)

/data/keeling/a/rytam2/a/iema_output/variables_202510210102.nc
